# Dimensionality Reduction Projection

This notebook will demonstrate the clean usage of `TopoVNL` based on the `MIMIC-MJX` package.

In [ ]:
import os
os.chdir('..')

# Step 1: Imports From Analysis Module & Checkpoints

In [ ]:
%env MUJOCO_GL=egl
%env PYOPENGL_PLATFORM=egl
%matplotlib inline
%load_ext autoreload
%autoreload 2

import numpy as np
import multiprocessing as mp
import functools
from tqdm import tqdm

from vnl_tda.utils.load_utils import (
    get_aggregate_data,
)

from vnl_tda.utils.rendering import (
    display_video
)

from vnl_tda.pipelines.animate_reduction.rendering_manifold import (
    render_pca_with_clips,
    render_umap_with_clips
)

In [ ]:
# external source
rodent_rollout_dir = "/root/vast/kaiwen/track-mjx/rollouts/rodent_force_analysis_rollout"
fly_rollout_dir = "/root/vast/kaiwen/track-mjx/rollouts/fly_force_analysis_rollout"
num_clips = 800

# Step 2: Loading & Preprocessing

We can visualize the gait cycle of each of the virtuak rodent, just like activation, we can load in the  `joint forces` and `intention` for each first.

In [ ]:
work_rodent = functools.partial(get_aggregate_data, "/sensor_readings", [], path=rodent_rollout_dir)
work_fly = functools.partial(get_aggregate_data, "/sensor_readings", [], path=fly_rollout_dir)

with mp.Pool(processes=mp.cpu_count()) as pool:
    jf_rodent = list(tqdm(pool.imap(work_rodent, range(num_clips)), total=num_clips))
    jf_fly = list(tqdm(pool.imap(work_fly, range(num_clips)), total=num_clips))

jf_rodent = np.vstack(jf_rodent)
jf_rodent = jf_rodent.reshape([num_clips, 499, 16])
jf_rodent = jf_rodent[:,:,9:13]

jf_fly = np.vstack(jf_fly)
jf_fly = jf_fly.reshape([num_clips, 999, 33]) # irrelevant (3 + 3 + 3) / force sensor (6x3) / touch sensor (6x1) = 33
jf_fly = jf_fly[:,:599,:]
jf_fly_force_sensor = jf_fly[:,:,9:27].reshape([num_clips, 599, 6, 3])
jf_fly = jf_fly_force_sensor[:,:,:,2].reshape([num_clips, 599, 6])

Some sanity check for the data quality of the forces

In [ ]:
np.argwhere(np.isnan(jf_rodent))

In [ ]:
np.argwhere(np.isnan(jf_rodent.reshape(-1, 4)))

In [ ]:
force_index = np.unique(np.nonzero(jf_rodent[0])[1])
force_index

In [ ]:
force_index = np.unique(np.nonzero(jf_fly[0])[1])
force_index

`Note` that we will be using teh 60 dimension intentionchecpoiint for all TDA as tehy seem to be more mature in forming their representation of gait cycle and the 16 dimension one for gait analysis as we have more sensors there.
- Is it due to less training in teh 16 dimension one that representation is unclear?

In [ ]:
work_fly = functools.partial(get_aggregate_data, "/activations", ['intention'], path=fly_rollout_dir)
work_rodent = functools.partial(get_aggregate_data, "/activations", ['intention'], path=rodent_rollout_dir)

with mp.Pool(processes=mp.cpu_count()) as pool:
    int_fly = list(tqdm(pool.imap(work_fly, range(num_clips)), total=num_clips))
    int_rodent = list(tqdm(pool.imap(work_rodent, range(num_clips)), total=num_clips))

int_rodent = np.vstack(int_rodent)
int_fly = np.vstack(int_fly)
int_fly = int_fly.reshape([num_clips, 999, 60])
int_fly = int_fly[:,:599,:]

Kicking out bad performance clips identified by topological features

In [ ]:
high_H1_clips = np.array([25, 97, 130, 134, 135, 197, 271, 315, 390, 415, 470, 525, 598, 610, 612, 621, 624, 694, 772, 784])
keep_mask = np.ones(num_clips, dtype=bool)
keep_mask[high_H1_clips] = False
int_fly = int_fly[keep_mask]
jf_fly = jf_fly[keep_mask]

int_fly = int_fly.reshape(-1, 60)
print(f"Filtered number of clips: {int_fly.shape}, {jf_fly.shape}")

Similarly, some sanity check of the data quality.

In [ ]:
np.argwhere(np.isnan(int_rodent.reshape(-1, 499, 60)))

In [ ]:
np.argwhere(np.isnan(int_rodent))

# Step 3: Dimensionality Reduction Visualization

## Step 3.1: Mujoco Clip Rollout Rendering With PCA Rendering

Note that this code may take more than 3 minutes to run due to the computationalcomplexity of rendering in mujoco and doing a global PCA with all the data.

In [ ]:
int_rodent = int_rodent.reshape(-1, 60)
int_fly = int_fly.reshape(-1, 60)
print(int_rodent.shape, jf_rodent.shape, int_fly.shape, jf_fly.shape)

In [ ]:
frames = render_pca_with_clips(int_array=int_fly,
                               rollout_dir=fly_rollout_dir,
                               clip_id=0,
                               clip_length=599,
                               walker_name="fly")

display_video(frames, framerate=50)

## Step 3.2: Mujoco Clip Rollout Rendering With UMAP Rendering

We can see one in the `fly` data that has gait cycle representation locally

In [ ]:
frames = render_umap_with_clips(int_array=int_fly,
                                 rollout_dir=fly_rollout_dir,
                                 clip_id=2,
                                 clip_length=599,
                                 walker_name="fly",
                                 window_size=50,
                                 stride=10)

display_video(frames, framerate=50)

In [ ]:
frames = render_umap_with_clips(int_array=int_rodent,
                                 rollout_dir=rodent_rollout_dir,
                                 clip_id=17,
                                 clip_length=499,
                                 walker_name="rodent",
                                 window_size=50,
                                 stride=10)

display_video(frames, framerate=50)

## Step 3.3: LSTM Decoder Comparison

In [ ]:
rodent_rollout_dir_lstm = "/root/vast/kaiwen/track-mjx/rollouts/rodent_force_analysis_rollout_lstm"
num_clips_lstm = 640

work_rodent_lstm = functools.partial(get_aggregate_data, "/activations", ['hidden_state'], path=rodent_rollout_dir_lstm)
work_rodent_lstm_proj = functools.partial(get_aggregate_data, "/activations", ['decoder','lstm_projection'], path=rodent_rollout_dir_lstm)
work_rodent_mlp = functools.partial(get_aggregate_data, "/activations", ['decoder','layer_2'], path=rodent_rollout_dir)

with mp.Pool(processes=mp.cpu_count()) as pool:
    hidden_cell_rodent = list(tqdm(pool.imap(work_rodent_lstm, range(num_clips_lstm)), total=num_clips_lstm))
    lstm_proj_rodent = list(tqdm(pool.imap(work_rodent_lstm_proj, range(num_clips_lstm)), total=num_clips_lstm))
    decoder_rodent = list(tqdm(pool.imap(work_rodent_mlp, range(num_clips_lstm)), total=num_clips_lstm))

hidden_cell_rodent = np.vstack(hidden_cell_rodent)
lstm_proj_rodent = np.vstack(lstm_proj_rodent)
decoder_rodent = np.vstack(decoder_rodent)

In [5]:
cell_rodent = hidden_cell_rodent[:,:,:,1,:]
hidden_rodent = hidden_cell_rodent[:,:,:,0,:]

We can do it with a mujoco clip rendering version, we will be comparing `mlp_decoder activations` with `lstm_hidden_state` and `lstm_cell_state`:

In [ ]:
frames = render_umap_with_clips(int_array=decoder_rodent,
                                 rollout_dir=rodent_rollout_dir,
                                 clip_id=3,
                                 clip_length=499,
                                 walker_name="rodent",
                                 window_size=60,
                                 stride=10)

display_video(frames, framerate=50)

In [ ]:
frames = render_umap_with_clips(int_array=hidden_rodent,
                                 rollout_dir=rodent_rollout_dir_lstm,
                                 clip_id=2,
                                 clip_length=499,
                                 walker_name="rodent",
                                 window_size=60,
                                 stride=10)

display_video(frames, framerate=50)